In [ ]:
import sys
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import decoupler as dc

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs
fs.CELL_COLORS = fs.HUMAN_CELL_COLORS

BASE = Path.cwd()
RESULTS = BASE / "results"
OUT_DIR = RESULTS / "04_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
PANELS = {
    "5a": (77, 62),
    "5b": (100, 62),
    "5c": (90, 70),
    "5d": (90, 70),
    "5Sa": (47, 84),
    "5Sb": (130, 84),
    "5Sc": (45, 50),
    "5Sd": (45, 50),
    "5Se": (45, 50),
    "5Sf": (45, 50),
}
fs.DPI = 500
plt.rcParams.update({
    "figure.dpi": fs.DPI,
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6, "figure.titlesize": 6, "legend.title_fontsize": 6,
    "xtick.labelsize": 5.5, "ytick.labelsize": 5.5, "legend.fontsize": 5.5,
})
SUPP_PT = 7

def panel_size(panel):
    width, height = PANELS[panel]
    return width * fs.MM, height * fs.MM

def save_panel(fig, name, tight=False):
    with plt.rc_context({"savefig.bbox": "tight" if tight else "standard", "savefig.pad_inches": 0}):
        fs.save(fig, OUT_DIR / name)

def font7(fig):
    for ax in fig.axes:
        ax.tick_params(labelsize=SUPP_PT)
    for text in fig.findobj(mpl.text.Text):
        text.set_fontsize(SUPP_PT)
    return fig

def fit_panel(draw, panel):
    target = np.array(panel_size(panel))
    size = target.copy()
    for _ in range(3):
        fig = draw(*size)
        size = size - (fig.get_tightbbox(fig.canvas.get_renderer()).size - target)
        plt.close(fig)
    return draw(*size)

def tidy_dotplot(dp):
    axes = dp.get_axes()
    size_ax, color_ax = axes["size_legend_ax"], axes["color_legend_ax"]
    size_ax.set_xticklabels([t.get_text() if i % 2 == 0 else "" for i, t in enumerate(size_ax.get_xticklabels())])
    for dots in size_ax.collections:
        dots.set_sizes([v if i % 2 == 0 else 0 for i, v in enumerate(dots.get_sizes())])
    fig_w, fig_h = dp.fig.get_size_inches()
    c, s = color_ax.get_position(), size_ax.get_position()
    color_ax.set_position([c.x0, c.y0, c.width, 0.08 / fig_h])
    size_ax.set_position([s.x0, c.y0 + (0.08 + 0.40) / fig_h, s.width, 0.12 / fig_h])
    size_ax.set_ylim(-0.6, 0.6)
    size_ax.tick_params(axis="x", length=0, pad=1)
    size_ax.set_title(size_ax.get_title(), y=1.0, pad=2, size="small")
    right = min(s.x0, c.x0) - 0.15 / fig_w
    for k in ["mainplot_ax", "gene_group_ax"]:
        if k in axes:
            p = axes[k].get_position()
            axes[k].set_position([p.x0, p.y0, right - p.x0, p.height])
    ax = axes["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    return ax

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_human_annotated.zarr")
obs = adata.obs
ROIS = ["SF1", "SF2", "Xray_Under", "Xray_Late", "Cion_Late"]

def volcano(results_df, a, b, panel, top_genes):
    fig, ax = plt.subplots(figsize=panel_size(panel), layout="constrained")
    dc.pl.volcano(results_df.dropna(subset=["padj"]), x="log2FoldChange", y="padj",
                  top=top_genes,
                  color_pos=fs.UP, color_neg=fs.DOWN, color_null=fs.LIGHTGREY,
                  kw_scatter={"s": 4, "lw": 0, "rasterized": True}, ax=ax)
    for line in ax.lines:
        line.set(color=fs.GREY, lw=0.4, ls=(0, (2, 2)))
    for text in ax.texts:
        text.set(fontsize=5, fontstyle="italic")
    ax.set_xlabel("log$_2$ fold change")
    ax.set_ylabel("−log$_{10}$(adjusted $\\it{P}$ value)")
    fig.legend(handles=[Line2D([], [], ls="", marker="o", ms=3, mew=0, color=fs.UP, label=f"Upregulated in {fs.label(a)}"),
                        Line2D([], [], ls="", marker="o", ms=3, mew=0, color=fs.DOWN, label=f"Upregulated in {fs.label(b)}")],
               loc="outside lower center", ncol=1, handletextpad=0.2, labelspacing=0.2)
    return fig

def read_deg(path):
    results_df = pd.read_csv(path, index_col="gene")
    pct = [c for c in results_df.columns if c.startswith("pct_") and c != "pct_nz_group"]
    return results_df[results_df[pct].max(axis=1) >= 0.1]

In [ ]:
def umap(ax, xy, labels, size=0.3):
    labels = pd.Categorical(labels).remove_unused_categories()
    cats = labels.categories
    order = np.random.default_rng(0).permutation(len(xy))
    colors = np.array(fs.palette(labels.astype(str)))
    ax.scatter(xy[order, 0], xy[order, 1], c=colors[order], s=size, lw=0, rasterized=True)
    ax.set_aspect("equal")
    ax.set_axis_off()

    handles = [mpl.lines.Line2D([], [], ls="", marker="o", ms=3, mew=0, color=c) for c in fs.palette(cats)]
    ax.legend(handles, cats, loc="center left", bbox_to_anchor=(1.0, 0.5), borderaxespad=0, handletextpad=0.1,
              labelspacing=0.25)

    span = np.ptp(xy, axis=0).max()
    x0, y0 = xy.min(axis=0) - 0.03 * span
    arrow = dict(arrowstyle="-|>", lw=0.5, color=fs.BLACK, mutation_scale=4, shrinkA=0, shrinkB=0)
    ax.update_datalim([(x0, y0)])
    ax.annotate("", (x0 + 0.15 * span, y0), (x0, y0), arrowprops=arrow)
    ax.annotate("", (x0, y0 + 0.15 * span), (x0, y0), arrowprops=arrow)
    ax.annotate("UMAP1", (x0 + 0.075 * span, y0), xytext=(0, -2), textcoords="offset points", fontsize=5,
                ha="center", va="top")
    ax.annotate("UMAP2", (x0, y0 + 0.075 * span), xytext=(-2, 0), textcoords="offset points", fontsize=5,
                ha="right", va="center", rotation=90)
    return ax

LEFT_CUT = 3 * fs.MM
width, height = panel_size("5a")
fig, ax = plt.subplots(figsize=(width + LEFT_CUT, height), layout="constrained")
umap(ax, adata.obsm["X_umap"], obs["cell_largetype"], size=0.3)
fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01)
crop = mpl.transforms.Bbox.from_bounds(LEFT_CUT, 0, width, height)
for ext in ("pdf", "png"):
    fig.savefig(OUT_DIR / f"5a_umap.{ext}", dpi=fs.DPI, bbox_inches=crop)
plt.show()

In [ ]:
def composition(obs, group, stack, groups, title=None, width=fs.COL2, height=55 * fs.MM, panels=("count", "fraction"),
                rows=None, row_labels=None, tick_label=None):
    ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]
    frac = ct.div(ct.sum(axis=1), axis=0) * 100
    spec = {"count": (ct, "Number of cells"), "fraction": (frac, "Proportion of cells (%)")}
    rows = rows or [groups]
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(width, height), layout="constrained", squeeze=False,
                             sharey="col")
    for r, row in enumerate(rows):
        for ax, panel in zip(axes[r], panels):
            table, ylabel = spec[panel]
            table = table.loc[row]
            xs = np.arange(len(table))
            bottom = np.zeros(len(table))
            for c, color in zip(table.columns, fs.palette(table.columns)):
                ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
                bottom += table[c].to_numpy(dtype=float)
            if tick_label is None:
                ax.set_xticks(xs, [fs.label(g) for g in table.index], rotation=45, ha="right", rotation_mode="anchor")
            else:
                ax.set_xticks(xs, [tick_label(g) for g in table.index])
            if row_labels is not None:
                ax.set_title(row_labels[r])
            ax.set_ylabel(ylabel)
            if panel == "count":
                ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
            else:
                ax.set_ylim(0, 100)
    handles, names = axes[0, -1].get_legend_handles_labels()
    fig.legend(handles[::-1], names[::-1], title=title, loc="outside right center", alignment="left")
    return fig, ct

fig, counts = composition(obs, "roi_name", "cell_largetype", ROIS, width=panel_size("5b")[0], height=panel_size("5b")[1],
                             panels=("count",))
save_panel(fig, "5b_largetype_count")
plt.show()
counts

In [ ]:
LABEL = ["CCN2", "SERPINE1", "LTBP2", "ELN", "FBLN5", "MFAP5", "COL5A1", "COL5A2", "ADAMTS1", "SNAI2", "CDKN1A", "SMOC2"]
deg_late = read_deg(RESULTS / "02_fibroblast_deg" / "fibroblast_deg_Xray_Late_vs_Cion_Late.csv")
fig = volcano(deg_late, "Xray_Late", "Cion_Late", "5c", LABEL)
save_panel(fig, "5c_fibroblast_volcano_Xray_Late_vs_Cion_Late")
plt.show()

In [ ]:
go = pd.read_csv(RESULTS / "02_fibroblast_deg" / "fibroblast_go_up_Xray_Late.csv")
top = go[go["source"] == "GO:BP"].head(10).iloc[::-1]
names = ["\n".join(textwrap.wrap(n, 42)) for n in top["name"]]
fig, ax = plt.subplots(figsize=panel_size("5d"), layout="constrained")
norm = plt.Normalize(top["p_value"].min(), top["p_value"].max())
ax.barh(range(len(top)), top["intersection_size"], color=plt.cm.viridis_r(norm(top["p_value"])), height=0.7)
exp = int(np.floor(np.log10(top["p_value"].max())))
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis_r"), ax=ax, shrink=0.6, aspect=15,
                  label=f"Adjusted $\\it{{P}}$ value (×10$^{{{exp}}}$)")
cb.outline.set_linewidth(0.5)
cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 10 ** exp:g}"))
ax.set_yticks(range(len(top)), names)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("Count")
save_panel(fig, "5d_go_bp")
plt.show()

In [ ]:
before = pd.read_parquet(RESULTS / "00_preprocess_and_cluster" / "qc_per_cell_before.parquet")
CUT = {"n_genes_by_counts": (5, None), "total_counts": (10, None)}
def qc_violin(ax, values, labels, cutoff, ylabel, colors):
    data = [np.log10(np.asarray(v, dtype=float) + 1) for v in values]
    xs = np.arange(1, len(data) + 1)
    parts = ax.violinplot(data, positions=xs, widths=0.8, showextrema=False)
    for body, color in zip(parts["bodies"], colors):
        body.set(facecolor=color, edgecolor=fs.BLACK, linewidth=0.3, alpha=1)
    ax.hlines([np.median(d) for d in data], xs - 0.25, xs + 0.25, color=fs.BLACK, lw=0.75)
    for c in cutoff:
        if c is not None:
            ax.axhline(np.log10(c + 1), color=fs.GREY, lw=0.6, ls=(0, (3, 2)))
    top = max(d.max() for d in data)
    ticks = [t for t in [0, 1, 10, 100, 1000, 10000, 100000] if np.log10(t + 1) <= top * 1.02]
    ax.set_yticks(np.log10(np.array(ticks) + 1), [f"{t:,}" for t in ticks])
    ax.set_xticks(xs, labels)
    ax.set_xlim(0.4, len(data) + 0.6)
    ax.set_ylabel(ylabel)

METRIC = {"total_counts": "Transcripts per cell", "n_genes_by_counts": "Genes per cell"}

fig, axes = plt.subplots(2, 1, figsize=panel_size("5Sa"), sharex=True, layout="constrained")
fig.get_layout_engine().set(h_pad=0.5 * fs.MM, hspace=0)
for ax, (col, ylabel) in zip(axes, METRIC.items()):
    qc_violin(ax, [before.loc[before["roi_name"] == r, col] for r in ROIS], [fs.label(r) for r in ROIS],
                 CUT[col], ylabel, colors=[fs.ROI_COLORS[r] for r in ROIS])
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.align_ylabels(axes)
font7(fig)
save_panel(fig, "5Sa_qc_violin")
plt.show()

In [ ]:
def cluster_dotplot(adata, markers, cluster, label=None, block=None, height=100 * fs.MM, width=fs.COL2, brackets=True,
                    dot_max=1, legend_width=1.5):
    obs = adata.obs
    names = list(markers)
    genes = {ct: [g for g in gs if g in adata.var_names] for ct, gs in markers.items()}
    genes = {ct: gs for ct, gs in genes.items() if gs}
    flat = [g for gs in genes.values() for g in gs]

    table = pd.DataFrame({"cluster": obs[cluster].astype(str)})
    table["block"] = obs[block].astype(str) if block is not None else ""
    table = table.drop_duplicates("cluster").set_index("cluster", drop=False).rename_axis(None)
    table["rank"] = table["block"].map(lambda b: names.index(b) if b in names else len(names))
    table["prefix"] = table["cluster"].str.replace(r"\d+$", "", regex=True)
    table["num"] = pd.to_numeric(table["cluster"].str.extract(r"(\d+)$")[0], errors="coerce")
    if label is None:
        x = adata[:, flat].layers["log1p_norm"]
        x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
        mean = pd.DataFrame(x, columns=flat).groupby(obs[cluster].astype(str).to_numpy()).mean()
        scaled = (mean - mean.min()) / (mean.max() - mean.min())
        score = pd.DataFrame({ct: scaled[gs].mean(axis=1) for ct, gs in genes.items()})
        table["rank"] = score.loc[table.index].to_numpy().argmax(axis=1)
        blocks = list(genes.values())
        table["top_gene"] = [flat.index(scaled.loc[c, blocks[r]].idxmax()) for c, r in zip(table.index, table["rank"])]
        table = table.sort_values(["rank", "top_gene", "prefix", "num", "cluster"])
    else:
        table = table.sort_values(["rank", "prefix", "num", "cluster"])
    rows = list(table["cluster"])

    key = "_cluster_label"
    obs[key] = pd.Categorical(obs[cluster].astype(str).map(dict(zip(table["cluster"], rows))), categories=rows)
    try:
        dp = sc.pl.dotplot(adata, genes if brackets else flat, groupby=key, layer="log1p_norm", standard_scale="var",
                           dot_min=0, dot_max=dot_max,
                           figsize=(width, height), show=False, return_fig=True)
        dp.style(largest_dot=40, dot_edge_lw=0.2)
        dp.legend(width=legend_width, size_title="% cells", colorbar_title="Scaled\nexpression")
        dp.make_figure()
    finally:
        del obs[key]

    ax = dp.get_axes()["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    ax.set_ylabel("Cluster" if label is None else "Cell type")
    return dp

markers = {
    "Myofiber":         ["MYH2", "MYOT"],
    "Fibroblast":       ["PDGFRA", "DPT"],
    "MuSC":             ["PAX7"],
    "Perineurial cell": ["NGFR", "CLDN1"],
    "Schwann cell":     ["MPZ", "SOX10"],
    "Adipocyte":        ["ADIPOQ", "PLIN1"],
    "Endothelial cell": ["PECAM1"],
    "Arterial EC":      ["GJA5", "EFNB2", "PODXL"],
    "EC":               ["NOTCH4", "ITGA6"],
    "Venous EC":        ["CCL14", "PLVAP", "SELP"],
    "Lymphatic EC":     ["FLT4", "PROX1", "PDPN"],
    "Arterial SMC":     ["MYLK", "CSRP1", "PLN"],
    "Venous SMC":       ["SYNPO2", "ADAMTS4", "STEAP4"],
    "Pericyte":         ["RGS5", "NOTCH3"],
    "Macrophage":       ["CD163", "MRC1"],
    "Neutrophil":       ["CSF3R", "FPR1"],
    "T cell":           ["CD3E", "CD8A"],
    "Plasma cell":      ["MZB1", "XBP1", "CD79A"],
    "Mast cell":        ["KIT"],
}
assigned = adata[(obs["cell_subtype"] != "Unassigned").values].copy()
assigned.obs["cell_subtype"] = assigned.obs["cell_subtype"].cat.remove_unused_categories()

def draw(width, height):
    dp = cluster_dotplot(assigned, markers, "cell_subtype", label="cell_subtype", block="cell_subtype",
                            height=height, width=width, legend_width=0.55)
    tidy_dotplot(dp)
    return font7(dp.fig)

fig = fit_panel(draw, "5Sb")
save_panel(fig, "5Sb_marker_dotplot", tight=True)
plt.show()

In [ ]:
for panel, (a, b) in {"5Sc": ("Xray_Late", "SF"), "5Sd": ("Cion_Late", "SF"),
                      "5Se": ("Xray_Under", "SF"), "5Sf": ("Xray_Under", "Xray_Late")}.items():
    results_df = read_deg(RESULTS / "03_fibroblast_vs_SF" / f"deg_{a}_vs_{b}.csv")
    fig = volcano(results_df, a, b, panel, [])
    font7(fig)
    save_panel(fig, f"{panel}_volcano_{a}_vs_{b}")
    plt.show()